In [3]:
import pandas as pd

# Path to the CSV file
csv_path = r'C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\Example Excel file_working.csv'

# Read the CSV file into a DataFrame
df = pd.read_csv(csv_path)

# Display the DataFrame
print("Contents of the CSV file:")
print(df)


Contents of the CSV file:
       Title Inventory-ID Container Name  Location  Purity  \
0  0001_A_Ah  23-523-0001              A       222  99.10%   
1  0002_B_Be  23-523-0002              B       275  99.20%   
2  0003_C_Ce  23-523-0003              C       114  99.30%   
3  0004_D_De  23-523-0004              D       222  99.40%   
4  0005_E_Eh  23-523-0005              E       275  99.50%   
5  0006_F_Ef  23-523-0006              F       114  99.60%   
6  0007_G_Ge  23-523-0007              G       222  99.70%   
7  0008_H_Ha  23-523-0008              H       275  99.80%   
8  0009_I_Ih  23-523-0009              I       114  99.90%   

               Manufacturer  CAS-Number  Batch Number Opening Date  
0  Thermo Fisher Scientific   7440-39-3     123456789     01-02-24  
1             Sigma Aldrich   7440-39-4     123456789     01-02-24  
2                Alfa Aesar   7440-39-5     123456789     01-02-24  
3  Thermo Fisher Scientific   7440-39-6     123456789     01-02-24  
4       

In [4]:
#!/usr/bin/env python
import os
import time
import datetime
import csv
import json
import elabapi_python
from elabapi_python.rest import ApiException
import pandas as pd

#########################
#         CONFIG        #
#########################
API_HOST_URL = 'https://eln01-t.ca.hrz.tu-darmstadt.de/api/v2'
# replace with your api key
API_KEY = '23-b3f02a4d01ec10b4dcba33596866a2435a7baaffc6252f7e61b57ab88a20db5a28364e8a77dc50b2291523'
#########################
#      END CONFIG       #
#########################

# Configure the api client
configuration = elabapi_python.Configuration()
configuration.api_key['api_key'] = API_KEY
configuration.api_key_prefix['api_key'] = 'Authorization'
configuration.host = API_HOST_URL
configuration.debug = False
configuration.verify_ssl = False

# Create an instance of the API class
api_client = elabapi_python.ApiClient(configuration)
# Fix issue with Authorization header not being properly set by the generated lib
api_client.set_default_header(header_name='Authorization', header_value=API_KEY)

# Load the items API
items_api = elabapi_python.ItemsApi(api_client)

# Path to the CSV file
csv_path = r'C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\Example Excel file_working.csv'

# Read the CSV file into a DataFrame
df = pd.read_csv(csv_path)

# Category ID in eLab where the items will be created
category_id = 238  # Adjust this according to your category ID in eLab

# Base JSON for metadata
base_json = '''{
    "elabftw": {
        "display_main_text": false
    },
    "extra_fields": {
        "Inventory-ID": {
            "type": "text",
            "value": ""
        },
        "Container Number": {
            "type": "text",
            "value": ""
        },
        "Location": {
            "type": "text",
            "value": ""
        },
        "Purity": {
            "type": "text",
            "value": ""
        },
        "Manufacturer": {
            "type": "text",
            "value": ""
        },
        "CAS-Number": {
            "type": "text",
            "value": ""
        },
        "Batch Number": {
            "type": "text",
            "value": ""
        },
        "Opening Date": {
            "type": "text",
            "value": ""
        }
    }
}'''

metadata_as_json = json.loads(base_json)

# Loop over rows in the DataFrame (skipping the first row as it contains headers)
for index, row in df.iterrows():
    if index == 0:
        continue  # Skip the header row

    try:
        # Create an item in the correct category
        response = items_api.post_item_with_http_info(body={'category_id': category_id, 'title': row['Title']})
        # Extract the newly created item ID
        location_header_in_response = response[2].get('Location')
        print(f'The newly created item is here: {location_header_in_response}')
        item_id = int(location_header_in_response.split('/').pop())

        # Update metadata with values from the current row
        metadata_as_json['extra_fields']['Inventory-ID']['value'] = row['Inventory-ID']
        metadata_as_json['extra_fields']['Container Number']['value'] = row['Container Name']
        metadata_as_json['extra_fields']['Location']['value'] = row['Location']
        metadata_as_json['extra_fields']['Purity']['value'] = row['Purity']
        metadata_as_json['extra_fields']['Manufacturer']['value'] = row['Manufacturer']
        metadata_as_json['extra_fields']['CAS-Number']['value'] = row['CAS-Number']
        metadata_as_json['extra_fields']['Batch Number']['value'] = row['Batch Number']
        metadata_as_json['extra_fields']['Opening Date']['value'] = row['Opening Date']

        # Modify the item with the correct metadata
        #items_api.patch_item(item_id, body={'metadata': json.dumps(metadata_as_json)})
        items_api.patch_item(item_id, body={'title': row['Title'], 'metadata': json.dumps(metadata_as_json)})

        print(f"Item '{row['Title']}' created successfully.")
    except ApiException as e:
        print(f"An error occurred while creating item '{row['Title']}': {e}")


C:\ProgramData\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1056: InsecureRequestWarning: Unverified HTTPS request is being made to host 'eln01-t.ca.hrz.tu-darmstadt.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1056: InsecureRequestWarning: Unverified HTTPS request is being made to host 'eln01-t.ca.hrz.tu-darmstadt.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


The newly created item is here: https://eln01-t.ca.hrz.tu-darmstadt.de/api/v2/items/6272
Item '0002_B_Be' created successfully.


C:\ProgramData\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1056: InsecureRequestWarning: Unverified HTTPS request is being made to host 'eln01-t.ca.hrz.tu-darmstadt.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1056: InsecureRequestWarning: Unverified HTTPS request is being made to host 'eln01-t.ca.hrz.tu-darmstadt.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


The newly created item is here: https://eln01-t.ca.hrz.tu-darmstadt.de/api/v2/items/6273
Item '0003_C_Ce' created successfully.


C:\ProgramData\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1056: InsecureRequestWarning: Unverified HTTPS request is being made to host 'eln01-t.ca.hrz.tu-darmstadt.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1056: InsecureRequestWarning: Unverified HTTPS request is being made to host 'eln01-t.ca.hrz.tu-darmstadt.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


...

C:\ProgramData\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1056: InsecureRequestWarning: Unverified HTTPS request is being made to host 'eln01-t.ca.hrz.tu-darmstadt.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1056: InsecureRequestWarning: Unverified HTTPS request is being made to host 'eln01-t.ca.hrz.tu-darmstadt.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


The newly created item is here: https://eln01-t.ca.hrz.tu-darmstadt.de/api/v2/items/6278
Item '0008_H_Ha' created successfully.


C:\ProgramData\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1056: InsecureRequestWarning: Unverified HTTPS request is being made to host 'eln01-t.ca.hrz.tu-darmstadt.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1056: InsecureRequestWarning: Unverified HTTPS request is being made to host 'eln01-t.ca.hrz.tu-darmstadt.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


The newly created item is here: https://eln01-t.ca.hrz.tu-darmstadt.de/api/v2/items/6279
Item '0009_I_Ih' created successfully.


C:\ProgramData\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1056: InsecureRequestWarning: Unverified HTTPS request is being made to host 'eln01-t.ca.hrz.tu-darmstadt.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
